In [2]:
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm

In [7]:
df = pd.read_csv('최종_홍수취약성_랭킹.csv')

print("1. 반지하 주택(S1)과 하수도 인프라(S2, S3) 사이의 상관관계")
corr_s2, p_s2 = stats.pearsonr(df['S1_정규화'], df['S2_Vulnerability'])
corr_s3, p_s3 = stats.pearsonr(df['S1_정규화'], df['S3_Vulnerability'])

print(f"반지하(S1) vs 하수관거(S2) 상관계수: {corr_s2:+.4f} (P-value: {p_s2:.4f})")
print(f"반지하(S1) vs 빗물받이(S3) 상관계수: {corr_s3:+.4f} (P-value: {p_s3:.4f})")

1. 반지하 주택(S1)과 하수도 인프라(S2, S3) 사이의 상관관계
반지하(S1) vs 하수관거(S2) 상관계수: +0.1468 (P-value: 0.4839)
반지하(S1) vs 빗물받이(S3) 상관계수: -0.0991 (P-value: 0.6373)


두 인프라 모두 P-value가 0.05를 초과하여 유의미한 상관관계가 없다.
이는 취약 거주지(반지하)에 배수 인프라가 집중되지 않고 흩어져 있는 미스매치 상태를 보여준다.

In [10]:
print("2. 펌프장/저류조(AC)의 방어 효과 (독립표본 T-검정)")

# AC 지수 중앙값을 기준으로 두 그룹 분리
median_ac = df['AC_최종지수'].median()
high_ac_group = df[df['AC_최종지수'] >= median_ac]['V_최종지수']
low_ac_group  = df[df['AC_최종지수'] < median_ac]['V_최종지수']

# 두 집단 간 평균 차이 검정
t_stat_ac, p_val_ac = stats.ttest_ind(high_ac_group, low_ac_group)

print(f"펌프장/저류조 우수 그룹의 평균 홍수 취약성: {high_ac_group.mean():.4f}")
print(f"펌프장/저류조 미흡 그룹의 평균 홍수 취약성: {low_ac_group.mean():.4f}")
print(f"T-통계량: {t_stat_ac:.4f} (P-value: {p_val_ac:.4f})")

2. 펌프장/저류조(AC)의 방어 효과 (독립표본 T-검정)
펌프장/저류조 우수 그룹의 평균 홍수 취약성: 0.4278
펌프장/저류조 미흡 그룹의 평균 홍수 취약성: 0.6353
T-통계량: -2.6609 (P-value: 0.0140)


빗물 펌프장과 저류조 용량 확보는 피해를 막는 방어 효과를 가지고 있다.

In [11]:
print("3. 홍수 방재 정책의 핵심 타겟 도출 (OLS 다중회귀분석)")

# 독립변수(X)와 종속변수(Y) 설정
X = df[['표면유출_위험지수', 'S1_정규화', 'S2_Vulnerability', 'S3_Vulnerability', 'AC_최종지수']]
X = sm.add_constant(X) # 회귀식의 상수항(절편) 추가
Y = df['V_최종지수']

# statsmodels의 OLS(최소제곱법) 모델 적합
model = sm.OLS(Y, X).fit()

# 전체 통계표 출력 (R-squared, F-statistic, 변수별 P-value 등을 한눈에 확인)
print(model.summary())

3. 홍수 방재 정책의 핵심 타겟 도출 (OLS 다중회귀분석)
                            OLS Regression Results                            
Dep. Variable:                 V_최종지수   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                     441.6
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           5.79e-19
Time:                        16:12:44   Log-Likelihood:                 62.660
No. Observations:                  25   AIC:                            -113.3
Df Residuals:                      19   BIC:                            -106.0
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const

[결론 및 시사점 요약]
1. 모델의 설명력(R-squared)이 99.1%로 매우 강력한 예측력을 가진다.
2. 표면유출 위험지수(Coefficient: 0.5883)가 취약성을 높이는 가장 큰 요인이다.
3. 방어 인프라 중 하수관거(S2)나 빗물받이(S3)보다, 펌프장/저류조(AC, Coefficient: -0.8989)의 취약성 감소 효과가 압도적으로 뛰어남이 입증되었다.
4. 결론적으로 향후 방재 예산은 '표면유출 억제' 및 '저류/펌프 시설 확충'에 집중되어야 한다.